# Demo: Inspyro Engineering Units System

Este notebook demuestra el flujo principal del modulo de unidades:

1. Creacion de `Quantity` con constantes de ingenieria (`kN`, `MPa`, `kg`, `m`, `s`).
2. Conversiones dimensionalmente validas y manejo de incompatibilidades.
3. Serializacion para frontend (`variables_summary`) y metadata de unidades.
4. Formato para DOCX (runs con superindice/cursiva).
5. Conversion REST opcional con `POST /api/units/convert`.


### Paso 1: Preparacion e imports

Esta celda configura el path `backend/` y carga la API de unidades, serializacion, metadata y formateadores que usara el resto del notebook.


In [ ]:
# Si ejecutas desde la raiz del repo, agrega backend al PYTHONPATH.
import pathlib
import sys

repo_root = pathlib.Path.cwd()
backend_path = repo_root / "backend"
if backend_path.exists() and str(backend_path) not in sys.path:
    sys.path.insert(0, str(backend_path))

from librerias_propias.inspyro_units import *
from librerias_propias.inspyro_units.serialization import serialize_quantity
from librerias_propias.inspyro_units.metadata import get_metadata_for_unit, get_category_for_unit
from librerias_propias.inspyro_units.formatting import (
    format_quantity_unicode,
    format_quantity_latex,
    format_quantity_html,
    format_quantity_docx,
    build_docx_unit_runs,
)

print("inspyro_units loaded OK")


### Paso 2: Cantidades base y calculos iniciales

Esta celda crea magnitudes de ingenieria (`F`, `A`, `sigma`, `rho`, `w`) y muestra operaciones con unidades compuestas.


In [ ]:
# Cantidades base de ingenieria.
F = 14.5 * kN
A = 5800 * mm**2
sigma = (F / A).to(MPa)

rho_steel = 7850 * kg / m**3
g_load = 9.81 * m / s**2
w = (rho_steel * g_load).to(kN / m**3)

print(f"F  = {F:~P}")
print(f"A  = {A:~P}")
print(f"sigma = {sigma:~P}")
print(f"rho = {rho_steel:~P}")
print(f"w   = {w:~P}")


### Paso 3: Conversiones y validacion dimensional

Esta celda convierte unidades compatibles (momento y aceleracion) y demuestra que operaciones incompatibles lanzan un error esperado.


In [ ]:
# Conversiones y chequeo dimensional.
L = 6.0 * m
M = (F * L).to(kNm)
u = 3.2 * m / s
a = (u / (2.0 * s)).to(m / s**2)

print(f"M = {M:~P}")
print(f"a = {a:~P}")

try:
    _ = F + L
except Exception as exc:
    print("Dimensionality check OK (error esperado):", type(exc).__name__)


### Paso 4: Serializacion para frontend

Esta celda usa `serialize_quantity` para construir payloads JSON con metadata, categoria y `repr` para `variables_summary` y el grafo.


In [ ]:
# Serializacion para frontend (variables_summary / grafo runtime).
payload_sigma = serialize_quantity(sigma)
payload_w = serialize_quantity(w)

print("sigma category:", get_category_for_unit(sigma))
print("sigma metadata:", get_metadata_for_unit(sigma))
print("payload keys:", sorted(payload_sigma.keys()))

payload_sigma, payload_w


### Paso 5: Formato de salida

Esta celda compara formatos Unicode, LaTeX y HTML, y genera estructuras de runs DOCX para unidades con exponentes.


In [ ]:
# Formateadores para salida (unicode, latex, html, docx-runs).
samples = [
    3.2 * m / s**2,
    7850 * kg / m**3,
    14.5 * kN,
    25 * MPa,
]

for q in samples:
    print("-", format_quantity_unicode(q))
    print("  latex:", format_quantity_latex(q))
    print("  html :", format_quantity_html(q))

if "build_docx_unit_runs" not in globals():
    from librerias_propias.inspyro_units.formatting import build_docx_unit_runs

docx_payload = format_quantity_docx(3.2 * m / s**2)
docx_runs_density = build_docx_unit_runs("kg/m^3")
docx_runs_force = build_docx_unit_runs("kN")

docx_payload, docx_runs_density, docx_runs_force


### Paso 6: Demo DOCX opcional

Esta celda intenta escribir una linea con unidades al DOCX usando `doc_block` o `build_doc` cuando la API DOCX esta disponible en el kernel.


In [ ]:
# Demo opcional de Fase 6 dentro de Inspyro: exporta texto con unidades al DOCX.
doc_line = f"La fuerza es {F:~P} y el esfuerzo es {sigma:~P}"

if "doc_block" in globals():
    with doc_block(order=900) as doc:
        doc.text(doc_line)
    print("doc_block ejecutado: revisa el DOCX/PDF en el visor")
elif "build_doc" in globals():
    with build_doc(order=900) as doc:
        doc.text(doc_line)
    print("build_doc ejecutado: revisa el DOCX/PDF en el visor")
else:
    print("API DOCX no disponible en este kernel (esperado fuera de Inspyro)")


### Paso 7: Demo REST opcional

Esta celda llama `POST /api/units/convert` en `localhost:8000` para probar conversiones desde backend y manejar errores si el servicio no esta activo.


In [ ]:
# Demo opcional de Fase 7: conversion REST en backend.
import json
import urllib.error
import urllib.request

request_body = {
    "magnitude": 14.5,
    "from_unit": "kN",
    "to_unit": "lbf",
}

url = "http://localhost:8000/api/units/convert"
req = urllib.request.Request(
    url,
    data=json.dumps(request_body).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST",
)

try:
    with urllib.request.urlopen(req, timeout=5) as resp:
        data = json.loads(resp.read().decode("utf-8"))
    print("REST conversion OK")
    data
except urllib.error.HTTPError as exc:
    err_text = exc.read().decode("utf-8", errors="replace")
    print("REST conversion HTTP error:", exc.code, err_text)
except Exception as exc:
    print("REST conversion skipped:", exc)
    print("Tip: inicia backend en localhost:8000 para ejecutar esta celda.")


## Checklist de validacion en UI Inspyro

1. Ejecuta las celdas y confirma `Quantity` en `variables_summary`.
2. Abre la pestana **Variables** y verifica tarjetas con categoria y dimension.
3. Prueba conversion rapida por variable (por ejemplo `kN -> lbf`).
4. Abre el grafo de dependencias y confirma prioridad de metadata runtime.
5. Si usas `doc_block`, verifica en DOCX/PDF unidades en cursiva y exponentes en superindice.
